In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from homonym import step1, step2, step3, step4, step5, fix1, fix2, fix3, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed1_03_homonymous"

df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
#df = prepare_event_df(log_name=f"./dataset/{LOG_NAME}.csv")
llm_repetition = 10

fix_repetition = 3
current_df = df.copy()

iteration_history = []

for i in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f"      ITERATION {i+1} START")
    print(f" {'='*30}")
    print("\n>>> STEP 1: Heuristic Homonym Identification")
    flow_all, flow_filtered = step1.run_step1(current_df)

    if not flow_filtered:
        print(f"\n[INFO] No more potential homonyms identified at Iteration {i+1}.")
        print("Stopping the pipeline as the log is considered refined.")
        break 

    print("\n>>> STEP 2: Structural Homonym Candidate Matching")
    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

    print("\n>>> STEP 3: Homonym Candidate Disaggregation")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, homonym_candidates, flow_all, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP3, prompts.USER_PROMPT_HOMONYM_STEP3)

    print("\n>>> STEP 4: Structural Consistency Validation")
    res_s4 = step4.run_step4(llm, MODEL, llm_repetition, res_s3, current_df,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1, prompts.USER_PROMPT_HOMONYM_STEP4_1,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2, prompts.USER_PROMPT_HOMONYM_STEP4_2)

    print("\n>>> STEP 5: Final Homonym Selection")
    res_s5 = step5.run_step5(llm, MODEL, llm_repetition, res_s4, current_df, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP5, prompts.USER_PROMPT_HOMONYM_STEP5)

    # --- FIX 1 ~ 3: Refinement & Mapping ---
    print("\n>>> FIX 1: Restoration Path Generation")
    res_f1 = fix1.run_fix1(llm, MODEL, llm_repetition, res_s5, current_df, 
                           prompts.SYSTEM_PROMPT_HOMONYM_FIX1, prompts.USER_PROMPT_HOMONYM_FIX1)

    print("\n>>> FIX 2: Event-level Refinement")
    res_f2 = fix2.run_fix2(llm, MODEL, llm_repetition, res_s5, res_f1, current_df, 
                           prompts.SYSTEM_PROMPT_HOMONYM_FIX2, prompts.USER_PROMPT_HOMONYM_FIX2)

    print("\n>>> FIX 3: DataFrame Mapping")
    res_f3 = fix3.run_fix3(current_df, res_f2)

    print("\n>>> EVALUATION: Scoring & Activity Update")
    metrics, current_df = evaluation.run_evaluation(res_f3)
    
    iteration_history.append({'iteration': i+1, 'metrics': metrics})
    
    print(f"\n[Iteration {i+1} Complete]")



      ITERATION 1 START

>>> STEP 1: Heuristic Homonym Identification
>>> Running Step 1 
    - Total activities: 13
    - Potential homonym candidates: 9
    - Candidate Sample: {'activity': 'Check for completeness', 'predecessors': ['info received', 'review request received']..., 'successors': ['Request info', 'Perform checks']...}

>>> STEP 2: Structural Homonym Candidate Matching
>>> Running Step 2
Mapping Preview (Total: 8 activities flagged)
  - Homonym Label (Sample): Check for completeness
    └─ Candidate 1: ['Notify accept', 'Verification process']
    └─ Candidate 2: ['Verification process', 'notify reject']
    └─ Candidate 3: ['Notify accept', 'info received']
    └─ ... and 2 more combinations
  ... and 7 more target activities discovered.

>>> STEP 3: Homonym Candidate Disaggregation
>>> Running Step 3 with 10 repetitions...
Mapping Preview (Total: 3 groups)
  - Target: [Decision review]
    └─ Sample: ['Make decision', 'review request received'] (4 total groups)
  - Ta

KeyError: 'f1'

In [3]:
for i in range(2):
    print(f"\n{'='*30}")
    print(f"      ITERATION {i+1} START")
    print(f" {'='*30}")
    print("\n>>> STEP 1: Heuristic Homonym Identification")
    flow_all, flow_filtered = step1.run_step1(current_df)

    if not flow_filtered:
        print(f"\n[INFO] No more potential homonyms identified at Iteration {i+1}.")
        print("Stopping the pipeline as the log is considered refined.")
        break 

    print("\n>>> STEP 2: Structural Homonym Candidate Matching")
    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

    print("\n>>> STEP 3: Homonym Candidate Disaggregation")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, homonym_candidates, flow_all, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP3, prompts.USER_PROMPT_HOMONYM_STEP3)

    print("\n>>> STEP 4: Structural Consistency Validation")
    res_s4 = step4.run_step4(llm, MODEL, llm_repetition, res_s3, current_df,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1, prompts.USER_PROMPT_HOMONYM_STEP4_1,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2, prompts.USER_PROMPT_HOMONYM_STEP4_2)

    print("\n>>> STEP 5: Final Homonym Selection")
    res_s5 = step5.run_step5(llm, MODEL, llm_repetition, res_s4, current_df, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP5, prompts.USER_PROMPT_HOMONYM_STEP5)

    # --- FIX 1 ~ 3: Refinement & Mapping ---
    print("\n>>> FIX 1: Restoration Path Generation")
    res_f1 = fix1.run_fix1(llm, MODEL, llm_repetition, res_s5, current_df, 
                           prompts.SYSTEM_PROMPT_HOMONYM_FIX1, prompts.USER_PROMPT_HOMONYM_FIX1)

    print("\n>>> FIX 2: Event-level Refinement")
    res_f2 = fix2.run_fix2(llm, MODEL, llm_repetition, res_s5, res_f1, current_df, 
                           prompts.SYSTEM_PROMPT_HOMONYM_FIX2, prompts.USER_PROMPT_HOMONYM_FIX2)

    print("\n>>> FIX 3: DataFrame Mapping")
    res_f3 = fix3.run_fix3(current_df, res_f2)

    print("\n>>> EVALUATION: Scoring & Activity Update")
    metrics, current_df = evaluation.run_evaluation(res_f3)
    
    iteration_history.append({'iteration': i+1, 'metrics': metrics})
    
    print(f"\n[Iteration {i+1} Complete]")



      ITERATION 1 START

>>> STEP 1: Heuristic Homonym Identification
>>> Running Step 1 
    - Total activities: 11
    - Potential homonym candidates: 3
    - Candidate Sample: {'activity': 'Check for completeness', 'predecessors': ['info received', 'review request received']..., 'successors': ['Perform checks', 'Request info']...}

>>> STEP 2: Structural Homonym Candidate Matching
>>> Running Step 2
Mapping Preview (Total: 3 activities flagged)
  - Homonym Label (Sample): Check for completeness
    └─ Candidate 1: ['Request info', 'info received']
  ... and 2 more target activities discovered.

>>> STEP 3: Homonym Candidate Disaggregation
>>> Running Step 3 with 10 repetitions...
Mapping Preview (Total: 2 groups)
  - Target: [Check for completeness]
    └─ Sample: ['Request info', 'info received'] (1 total groups)
  - Target: [Information exchange]
    └─ Sample: ['Request info', 'info received'] (5 total groups)

>>> STEP 4: Structural Consistency Validation

>>> Running Step 4 wi

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from homonym import step1, step2, step3, step4, step5, fix1, fix2, fix3, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "pub_seed1_03_homonymous"

df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
#df = prepare_event_df(log_name=f"./dataset/{LOG_NAME}.csv")
llm_repetition = 10

fix_repetition = 3
current_df = df.copy()

iteration_history = []

for i in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f"      ITERATION {i+1} START")
    print(f" {'='*30}")
    print("\n>>> STEP 1: Heuristic Homonym Identification")
    flow_all, flow_filtered = step1.run_step1(current_df)

    if not flow_filtered:
        print(f"\n[INFO] No more potential homonyms identified at Iteration {i+1}.")
        print("Stopping the pipeline as the log is considered refined.")
        break 

    print("\n>>> STEP 2: Structural Homonym Candidate Matching")
    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

    print("\n>>> STEP 3: Homonym Candidate Disaggregation")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, homonym_candidates, flow_all, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP3, prompts.USER_PROMPT_HOMONYM_STEP3)

    print("\n>>> STEP 4: Structural Consistency Validation")
    res_s4 = step4.run_step4(llm, MODEL, llm_repetition, res_s3, current_df,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1, prompts.USER_PROMPT_HOMONYM_STEP4_1,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2, prompts.USER_PROMPT_HOMONYM_STEP4_2)

    print("\n>>> STEP 5: Final Homonym Selection")
    res_s5 = step5.run_step5(llm, MODEL, llm_repetition, res_s4, current_df, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP5, prompts.USER_PROMPT_HOMONYM_STEP5)

    # --- FIX 1 ~ 3: Refinement & Mapping ---
    print("\n>>> FIX 1: Restoration Path Generation")
    res_f1 = fix1.run_fix1(llm, MODEL, llm_repetition, res_s5, current_df, 
                           prompts.SYSTEM_PROMPT_HOMONYM_FIX1, prompts.USER_PROMPT_HOMONYM_FIX1)

    print("\n>>> FIX 2: Event-level Refinement")
    res_f2 = fix2.run_fix2(llm, MODEL, llm_repetition, res_s5, res_f1, current_df, 
                           prompts.SYSTEM_PROMPT_HOMONYM_FIX2, prompts.USER_PROMPT_HOMONYM_FIX2)

    print("\n>>> FIX 3: DataFrame Mapping")
    res_f3 = fix3.run_fix3(current_df, res_f2)

    print("\n>>> EVALUATION: Scoring & Activity Update")
    metrics, current_df = evaluation.run_evaluation(res_f3)
    
    iteration_history.append({'iteration': i+1, 'metrics': metrics})
    
    print(f"\n[Iteration {i+1} Complete]")



      ITERATION 1 START

>>> STEP 1: Heuristic Homonym Identification
>>> Running Step 1 
    - Total activities: 14
    - Potential homonym candidates: 7
    - Candidate Sample: {'activity': 'Bring drinks', 'predecessors': ['prepare drinks', 'Drink handling'], 'successors': ['Review drinks', 'Drink handling']}

>>> STEP 2: Structural Homonym Candidate Matching
>>> Running Step 2
Mapping Preview (Total: 7 activities flagged)
  - Homonym Label (Sample): Bring drinks
    └─ Candidate 1: ['Prepare main course', 'Prepare starter']
    └─ Candidate 2: ['Prepare main course', 'Review order']
    └─ Candidate 3: ['Prepare starter', 'Signal food complete']
    └─ ... and 2 more combinations
  ... and 6 more target activities discovered.

>>> STEP 3: Homonym Candidate Disaggregation
>>> Running Step 3 with 10 repetitions...
Mapping Preview (Total: 4 groups)
  - Target: [Drink handling]
    └─ Sample: ['Bring drinks', 'Review drinks', 'prepare drinks'] (2 total groups)
  - Target: [Food deliver

In [4]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from homonym import step1, step2, step3, step4, step5, fix1, fix2, fix3, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "pub_seed52_ratio0.3_synonymous_withLabel"

df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
#df = prepare_event_df(log_name=f"./dataset/{LOG_NAME}.csv")
llm_repetition = 10

fix_repetition = 1
current_df = df.copy()

iteration_history = []

for i in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f"      ITERATION {i+1} START")
    print(f" {'='*30}")
    print("\n>>> STEP 1: Heuristic Homonym Identification")
    flow_all, flow_filtered = step1.run_step1(current_df)

    if not flow_filtered:
        print(f"\n[INFO] No more potential homonyms identified at Iteration {i+1}.")
        print("Stopping the pipeline as the log is considered refined.")
        break 

    print("\n>>> STEP 2: Structural Homonym Candidate Matching")
    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

    print("\n>>> STEP 3: Homonym Candidate Disaggregation")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, homonym_candidates, flow_all, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP3, prompts.USER_PROMPT_HOMONYM_STEP3)

    print("\n>>> STEP 4: Structural Consistency Validation")
    res_s4 = step4.run_step4(llm, MODEL, llm_repetition, res_s3, current_df,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1, prompts.USER_PROMPT_HOMONYM_STEP4_1,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2, prompts.USER_PROMPT_HOMONYM_STEP4_2)

    print("\n>>> STEP 5: Final Homonym Selection")
    res_s5 = step5.run_step5(llm, MODEL, llm_repetition, res_s4, current_df, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP5, prompts.USER_PROMPT_HOMONYM_STEP5)

    
    print(f"\n[Iteration {i+1} Complete]")




      ITERATION 1 START

>>> STEP 1: Heuristic Homonym Identification
>>> Running Step 1 
    - Total activities: 40
    - Potential homonym candidates: 12
    - Candidate Sample: {'activity': 'Bring drinks', 'predecessors': ['prepare drinks', 'mix drinks']..., 'successors': ['Review drinks', 'check drinks']...}

>>> STEP 2: Structural Homonym Candidate Matching
>>> Running Step 2
Mapping Preview (Total: 10 activities flagged)
  - Homonym Label (Sample): Bring drinks
    └─ Candidate 1: ['carry drinks', 'deliver drinks']
    └─ Candidate 2: ['carry drinks', 'serve drinks']
    └─ Candidate 3: ['deliver drinks', 'serve drinks']
    └─ ... and 1 more combinations
  ... and 9 more target activities discovered.

>>> STEP 3: Homonym Candidate Disaggregation
>>> Running Step 3 with 10 repetitions...
Mapping Preview (Total: 8 groups)
  - Target: [Bring drinks]
    └─ Sample: ['carry drinks', 'deliver drinks'] (4 total groups)
  - Target: [Review drinks]
    └─ Sample: ['check drinks', 'inspe

In [5]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from homonym import step1, step2, step3, step4, step5, fix1, fix2, fix3, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed52_ratio0.3_synonymous_withLabel"

df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
#df = prepare_event_df(log_name=f"./dataset/{LOG_NAME}.csv")
llm_repetition = 10

fix_repetition = 1
current_df = df.copy()

iteration_history = []

for i in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f"      ITERATION {i+1} START")
    print(f" {'='*30}")
    print("\n>>> STEP 1: Heuristic Homonym Identification")
    flow_all, flow_filtered = step1.run_step1(current_df)

    if not flow_filtered:
        print(f"\n[INFO] No more potential homonyms identified at Iteration {i+1}.")
        print("Stopping the pipeline as the log is considered refined.")
        break 

    print("\n>>> STEP 2: Structural Homonym Candidate Matching")
    homonym_candidates = step2.run_step2(flow_all, flow_filtered)

    print("\n>>> STEP 3: Homonym Candidate Disaggregation")
    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, homonym_candidates, flow_all, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP3, prompts.USER_PROMPT_HOMONYM_STEP3)

    print("\n>>> STEP 4: Structural Consistency Validation")
    res_s4 = step4.run_step4(llm, MODEL, llm_repetition, res_s3, current_df,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_1, prompts.USER_PROMPT_HOMONYM_STEP4_1,
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP4_2, prompts.USER_PROMPT_HOMONYM_STEP4_2)

    print("\n>>> STEP 5: Final Homonym Selection")
    res_s5 = step5.run_step5(llm, MODEL, llm_repetition, res_s4, current_df, 
                             prompts.SYSTEM_PROMPT_HOMONYM_STEP5, prompts.USER_PROMPT_HOMONYM_STEP5)

    
    print(f"\n[Iteration {i+1} Complete]")



      ITERATION 1 START

>>> STEP 1: Heuristic Homonym Identification
>>> Running Step 1 
    - Total activities: 39
    - Potential homonym candidates: 26
    - Candidate Sample: {'activity': 'Check for completeness', 'predecessors': ['info received', 'review request received']..., 'successors': ['Request info', 'Perform checks']...}

>>> STEP 2: Structural Homonym Candidate Matching
>>> Running Step 2
Mapping Preview (Total: 26 activities flagged)
  - Homonym Label (Sample): Check for completeness
    └─ Candidate 1: ['confirm completeness', 'validate completeness']
    └─ Candidate 2: ['confirm completeness', 'verify completeness']
    └─ Candidate 3: ['validate completeness', 'verify completeness']
    └─ ... and 1 more combinations
  ... and 25 more target activities discovered.

>>> STEP 3: Homonym Candidate Disaggregation
>>> Running Step 3 with 10 repetitions...
Mapping Preview (Total: 26 groups)
  - Target: [Check for completeness]
    └─ Sample: ['confirm completeness', 'val